# ENNI Developmental Analysis: Half-Life

**Dataset:** ENNI (Edmonton Narrative Norms Instrument)
- **TD**: 285 typically-developing children (ages 4-9)
- **LI**: 75 children with language impairment
- **6 stories per child**: A1, A2, A3, B1, B2, B3

**Analysis:**
1. **Primary**: TD developmental curves (half-life) across age bins
2. **Secondary**: TD vs LI comparison

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

In [ ]:
# Configuration
CONFIG = {
    'dataset': 'enni',
    'version': 'v4.2_halflife',
    'model_primary': 'mistralai/Mistral-7B-v0.1',
    'model_fallback': 'Qwen/Qwen2.5-7B',
    'max_context': 512,
    'age_bins': ['4-5yr', '5-6yr', '6-7yr', '7-8yr', '8-9yr', '9-10yr'],
    'groups': ['TD', 'LI'],
}

print(f"Config: {CONFIG['dataset']} {CONFIG['version']}")

## Load Model

In [ ]:
def load_model(model_name, fallback_name=None):
    """Load model with quantization and fallback support."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    try:
        print(f"Loading {model_name}...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        actual_model = model_name
    except Exception as e:
        if fallback_name:
            print(f"Primary model failed ({e}), trying fallback: {fallback_name}")
            tokenizer = AutoTokenizer.from_pretrained(fallback_name)
            model = AutoModelForCausalLM.from_pretrained(
                fallback_name,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
            )
            actual_model = fallback_name
        else:
            raise e
    
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    print(f"Loaded: {actual_model}")
    return model, tokenizer, actual_model

model, tokenizer, actual_model_name = load_model(
    CONFIG['model_primary'],
    CONFIG['model_fallback']
)
CONFIG['actual_model'] = actual_model_name

## Load ENNI Data

In [ ]:
from google.colab import files, drive
import os

# Upload ENNI clean transcripts JSONL
print("Please upload enni_clean.jsonl")
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]

# Mount Drive for saving results
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/LRTIA/results/ENNI/v4.2_halflife'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load transcripts
records = []
with open(DATA_PATH) as f:
    for line in f:
        r = json.loads(line)
        # Filter UNK group
        if r.get('group') not in ('TD', 'LI'):
            continue
        records.append(r)

print(f"Loaded {len(records)} ENNI story documents")

# Add age_bin if not present
def get_age_bin(age_years):
    if age_years is None:
        return 'UNK'
    if age_years < 5: return '4-5yr'
    elif age_years < 6: return '5-6yr'
    elif age_years < 7: return '6-7yr'
    elif age_years < 8: return '7-8yr'
    elif age_years < 9: return '8-9yr'
    else: return '9-10yr'

for r in records:
    if 'age_bin' not in r:
        r['age_bin'] = get_age_bin(r.get('age_years'))

# Summary
df_records = pd.DataFrame(records)
print("\nGroup distribution:")
print(df_records.groupby('group').size())
print("\nTD age distribution:")
print(df_records[df_records['group'] == 'TD'].groupby('age_bin').size())

# Contamination check
print("\nContamination metrics:")
print(f"  Mean pct_markup: {df_records['pct_markup'].mean():.6f}")
print(f"  Max pct_markup: {df_records['pct_markup'].max():.6f}")
print(f"  Mean pct_digits: {df_records['pct_digits'].mean():.6f}")

## Core Analysis Functions

In [ ]:
# ============================================================
# END-OF-DOC HALF-LIFE ANALYSIS
# ============================================================

def get_context_lengths(max_context):
    """Generate context lengths with dense sampling at short range."""
    lengths = []
    lengths.extend(range(4, min(33, max_context + 1), 4))
    lengths.extend(range(48, min(129, max_context + 1), 16))
    lengths.extend(range(160, max_context + 1, 32))
    return sorted(set(lengths))


def compute_cumulative_min(contexts, perplexities):
    """Compute monotonic decreasing envelope."""
    ppl_mon = np.zeros_like(perplexities)
    ppl_mon[0] = perplexities[0]
    for i in range(1, len(perplexities)):
        ppl_mon[i] = min(ppl_mon[i-1], perplexities[i])
    return contexts, ppl_mon


def interpolate_perplexity(contexts, ppl_mon, target_ctx):
    """Interpolate perplexity at target context."""
    if target_ctx <= contexts[0]:
        return ppl_mon[0]
    if target_ctx >= contexts[-1]:
        return ppl_mon[-1]
    for i in range(len(contexts) - 1):
        if contexts[i] <= target_ctx <= contexts[i+1]:
            frac = (target_ctx - contexts[i]) / (contexts[i+1] - contexts[i])
            return ppl_mon[i] + frac * (ppl_mon[i+1] - ppl_mon[i])
    return ppl_mon[-1]


def compute_half_life_robust(contexts, perplexities, max_ctx_fixed=None, benefit_eps=1.0):
    """Compute half-life with cumulative-min envelope."""
    contexts, ppl_mon = compute_cumulative_min(contexts, perplexities)
    
    ppl_at_min = ppl_mon[0]
    max_ctx_available = contexts[-1]
    max_ctx_used = min(max_ctx_available, max_ctx_fixed) if max_ctx_fixed else max_ctx_available
    
    ppl_at_max = interpolate_perplexity(contexts, ppl_mon, max_ctx_used)
    total_benefit = ppl_at_min - ppl_at_max
    benefit_ok = total_benefit >= benefit_eps
    
    half_life = np.nan
    if benefit_ok:
        target_ppl = ppl_at_min - 0.5 * total_benefit
        for i in range(len(ppl_mon) - 1):
            if contexts[i+1] > max_ctx_used:
                break
            if ppl_mon[i] >= target_ppl >= ppl_mon[i+1]:
                frac = (ppl_mon[i] - target_ppl) / (ppl_mon[i] - ppl_mon[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    return {
        'half_life': half_life,
        'total_benefit': total_benefit,
        'ppl_at_min_ctx': ppl_at_min,
        'ppl_at_max_ctx': ppl_at_max,
        'max_ctx_used': max_ctx_used,
        'benefit_ok': benefit_ok,
    }


def analyze_end_of_doc(model, tokenizer, text, max_context=512):
    """Compute perplexity at varying context lengths (end-of-doc)."""
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=2048)
    input_ids = inputs['input_ids'].to(model.device)
    T = input_ids.shape[1]
    
    if T < 50:
        return None
    
    context_lengths = get_context_lengths(min(max_context, T - 20))
    perplexities = []
    
    with torch.no_grad():
        for ctx_len in context_lengths:
            if ctx_len >= T:
                break
            
            start_idx = T - ctx_len
            context_ids = input_ids[:, start_idx:]
            
            outputs = model(context_ids, labels=context_ids)
            ppl = torch.exp(outputs.loss).item()
            perplexities.append(ppl)
    
    if len(perplexities) < 5:
        return None
    
    contexts = np.array(context_lengths[:len(perplexities)])
    perplexities = np.array(perplexities)
    
    result = compute_half_life_robust(contexts, perplexities, max_ctx_fixed=max_context)
    result['T'] = T
    result['contexts'] = contexts.tolist()
    result['perplexities'] = perplexities.tolist()
    
    return result

## Run Primary Analysis: TD Developmental Curves

In [ ]:
# Filter TD records only for primary analysis
td_records = [r for r in records if r['group'] == 'TD']
print(f"TD records: {len(td_records)}")

# Results storage
td_results = []

for record in tqdm(td_records, desc="Processing TD docs"):
    doc_id = record['doc_id']
    text = record['text']
    age_bin = record['age_bin']
    story_id = record['story_id']
    
    result = {
        'doc_id': doc_id,
        'group': 'TD',
        'age_bin': age_bin,
        'story_id': story_id,
        'age_years': record['age_years'],
        'word_count': record.get('word_count', len(text.split())),
    }
    
    # End-of-doc half-life
    eod = analyze_end_of_doc(model, tokenizer, text, max_context=CONFIG['max_context'])
    if eod:
        result['half_life'] = eod['half_life']
        result['total_benefit'] = eod['total_benefit']
        result['ppl_at_min_ctx'] = eod['ppl_at_min_ctx']
        result['ppl_at_max_ctx'] = eod['ppl_at_max_ctx']
        result['T'] = eod['T']
    
    td_results.append(result)
    
    # Periodic save
    if len(td_results) % 100 == 0:
        df_temp = pd.DataFrame(td_results)
        df_temp.to_csv(f'{RESULTS_DIR}/enni_td_partial.csv', index=False)

# Save final TD results
df_td = pd.DataFrame(td_results)
df_td.to_csv(f'{RESULTS_DIR}/enni_td_doc_results.csv', index=False)
print(f"\nSaved {len(df_td)} TD results")

## Run Secondary Analysis: LI Comparison

In [ ]:
# LI records
li_records = [r for r in records if r['group'] == 'LI']
print(f"LI records: {len(li_records)}")

li_results = []

for record in tqdm(li_records, desc="Processing LI docs"):
    doc_id = record['doc_id']
    text = record['text']
    age_bin = record['age_bin']
    story_id = record['story_id']
    
    result = {
        'doc_id': doc_id,
        'group': 'LI',
        'age_bin': age_bin,
        'story_id': story_id,
        'age_years': record['age_years'],
        'word_count': record.get('word_count', len(text.split())),
    }
    
    # End-of-doc half-life
    eod = analyze_end_of_doc(model, tokenizer, text, max_context=CONFIG['max_context'])
    if eod:
        result['half_life'] = eod['half_life']
        result['total_benefit'] = eod['total_benefit']
        result['ppl_at_min_ctx'] = eod['ppl_at_min_ctx']
        result['ppl_at_max_ctx'] = eod['ppl_at_max_ctx']
        result['T'] = eod['T']
    
    li_results.append(result)

# Save LI results
df_li = pd.DataFrame(li_results)
df_li.to_csv(f'{RESULTS_DIR}/enni_li_doc_results.csv', index=False)
print(f"\nSaved {len(df_li)} LI results")

# Combine all results
df_all = pd.concat([df_td, df_li], ignore_index=True)
df_all.to_csv(f'{RESULTS_DIR}/enni_all_doc_results.csv', index=False)
print(f"Total: {len(df_all)} documents")

## Aggregate Results by Age Bin

In [ ]:
def bootstrap_ci(values, n_boot=1000, ci=0.95):
    """Compute bootstrap confidence interval."""
    values = np.array([v for v in values if not np.isnan(v)])
    if len(values) < 3:
        return np.nan, np.nan, np.nan
    
    boot_means = [np.mean(np.random.choice(values, len(values), replace=True)) for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.mean(values), np.percentile(boot_means, alpha*100), np.percentile(boot_means, (1-alpha)*100)


def aggregate_by_group(df, group_cols, metrics):
    """Aggregate results by group with bootstrap CIs."""
    results = []
    
    for name, group_df in df.groupby(group_cols):
        if isinstance(name, tuple):
            row = dict(zip(group_cols, name))
        else:
            row = {group_cols[0]: name}
        
        row['n'] = len(group_df)
        
        for metric in metrics:
            if metric in group_df.columns:
                values = group_df[metric].dropna().values
                if len(values) > 0:
                    mean, ci_low, ci_high = bootstrap_ci(values)
                    row[f'{metric}_mean'] = mean
                    row[f'{metric}_ci_low'] = ci_low
                    row[f'{metric}_ci_high'] = ci_high
                    row[f'{metric}_n'] = len(values)
        
        results.append(row)
    
    return pd.DataFrame(results)


# Metrics to aggregate
metrics = ['half_life', 'total_benefit', 'ppl_at_min_ctx', 'ppl_at_max_ctx']

# TD by age_bin
df_td_by_age = aggregate_by_group(df_td, ['age_bin'], metrics)
df_td_by_age = df_td_by_age.sort_values('age_bin', key=lambda x: x.map({b: i for i, b in enumerate(CONFIG['age_bins'])}))
df_td_by_age.to_csv(f'{RESULTS_DIR}/enni_td_by_agebin.csv', index=False)
print("TD by age_bin:")
print(df_td_by_age[['age_bin', 'n', 'half_life_mean', 'half_life_ci_low', 'half_life_ci_high', 'total_benefit_mean']].to_string(index=False))

# TD by age_bin x story_id
df_td_by_story = aggregate_by_group(df_td, ['age_bin', 'story_id'], metrics)
df_td_by_story.to_csv(f'{RESULTS_DIR}/enni_td_by_agebin_story.csv', index=False)

# TD vs LI by age_bin
df_by_group_age = aggregate_by_group(df_all, ['group', 'age_bin'], metrics)
df_by_group_age.to_csv(f'{RESULTS_DIR}/enni_by_group_agebin.csv', index=False)
print("\nTD vs LI by age_bin:")
print(df_by_group_age[['group', 'age_bin', 'n', 'half_life_mean', 'total_benefit_mean']].to_string(index=False))

## Visualizations

In [ ]:
# Plot TD developmental curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

age_order = CONFIG['age_bins']
df_plot = df_td_by_age.set_index('age_bin').reindex(age_order)

# Half-life
ax = axes[0]
ax.errorbar(range(len(age_order)), df_plot['half_life_mean'],
            yerr=[df_plot['half_life_mean'] - df_plot['half_life_ci_low'],
                  df_plot['half_life_ci_high'] - df_plot['half_life_mean']],
            marker='o', capsize=5)
ax.set_xticks(range(len(age_order)))
ax.set_xticklabels(age_order, rotation=45)
ax.set_xlabel('Age Group')
ax.set_ylabel('Half-life (tokens)')
ax.set_title('TD: Half-life by Age')
ax.grid(True, alpha=0.3)

# Total benefit
ax = axes[1]
ax.errorbar(range(len(age_order)), df_plot['total_benefit_mean'],
            yerr=[df_plot['total_benefit_mean'] - df_plot['total_benefit_ci_low'],
                  df_plot['total_benefit_ci_high'] - df_plot['total_benefit_mean']],
            marker='o', capsize=5, color='green')
ax.set_xticks(range(len(age_order)))
ax.set_xticklabels(age_order, rotation=45)
ax.set_xlabel('Age Group')
ax.set_ylabel('Total Benefit (perplexity reduction)')
ax.set_title('TD: Context Benefit by Age')
ax.grid(True, alpha=0.3)

# TD vs LI comparison (half-life)
ax = axes[2]
for group, color in [('TD', 'blue'), ('LI', 'red')]:
    df_g = df_by_group_age[df_by_group_age['group'] == group].set_index('age_bin').reindex(age_order)
    ax.errorbar(range(len(age_order)), df_g['half_life_mean'],
                yerr=[df_g['half_life_mean'] - df_g['half_life_ci_low'],
                      df_g['half_life_ci_high'] - df_g['half_life_mean']],
                marker='o', capsize=5, color=color, label=group)
ax.set_xticks(range(len(age_order)))
ax.set_xticklabels(age_order, rotation=45)
ax.set_xlabel('Age Group')
ax.set_ylabel('Half-life (tokens)')
ax.set_title('TD vs LI: Half-life by Age')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/enni_developmental_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RESULTS_DIR}/enni_developmental_curves.png")

## Summary Statistics

In [ ]:
print("="*70)
print("ENNI ANALYSIS SUMMARY")
print("="*70)
print(f"Model: {CONFIG['actual_model']}")
print(f"Dataset: {CONFIG['dataset']} {CONFIG['version']}")
print(f"TD documents: {len(df_td)}")
print(f"LI documents: {len(df_li)}")

print("\n" + "="*70)
print("TD DEVELOPMENTAL TRENDS")
print("="*70)

# Compute correlations with age
age_numeric = df_td['age_bin'].map({b: i for i, b in enumerate(CONFIG['age_bins'])})

for metric in ['half_life', 'total_benefit']:
    if metric in df_td.columns:
        valid = df_td[[metric]].join(age_numeric.rename('age_num')).dropna()
        if len(valid) > 10:
            r, p = stats.spearmanr(valid['age_num'], valid[metric])
            print(f"{metric:25s}: r={r:+.3f}, p={p:.4f} {'*' if p < 0.05 else ''}")

print("\n" + "="*70)
print("TD vs LI COMPARISON (pooled across ages)")
print("="*70)

for metric in ['half_life', 'total_benefit']:
    if metric in df_all.columns:
        td_vals = df_all[df_all['group'] == 'TD'][metric].dropna()
        li_vals = df_all[df_all['group'] == 'LI'][metric].dropna()
        if len(td_vals) > 5 and len(li_vals) > 5:
            stat, p = stats.mannwhitneyu(td_vals, li_vals, alternative='two-sided')
            print(f"{metric:25s}: TD={td_vals.mean():.2f}, LI={li_vals.mean():.2f}, p={p:.4f} {'*' if p < 0.05 else ''}")

In [ ]:
# Save config
config_out = {
    **CONFIG,
    'n_td_docs': len(df_td),
    'n_li_docs': len(df_li),
    'results_dir': RESULTS_DIR,
}
with open(f'{RESULTS_DIR}/config.json', 'w') as f:
    json.dump(config_out, f, indent=2)

print(f"\nAll results saved to: {RESULTS_DIR}")
print("Files:")
print("  - enni_td_doc_results.csv")
print("  - enni_li_doc_results.csv")
print("  - enni_all_doc_results.csv")
print("  - enni_td_by_agebin.csv")
print("  - enni_td_by_agebin_story.csv")
print("  - enni_by_group_agebin.csv")
print("  - enni_developmental_curves.png")
print("  - config.json")